# 经典 CNN 架构：VGG 与 ResNet

前面我们自己搭了一个简单 CNN，在 CIFAR-10 上达到了 ~70-80% 准确率。现在看看学术界是怎么设计更好的 CNN 的。

### 为什么要看经典架构？

自己随便堆层不一定好——太深会梯度消失，太浅特征不够。VGG 和 ResNet 解决了这两个问题，是后续所有视觉模型的基础。

In [1]:
import torch
import torch.nn as nn

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"设备: {device}")

设备: cuda


## 1. VGG 的核心思想

### 问题：大卷积核 vs 小卷积核堆叠

一个 5x5 卷积核的感受野是 5x5。但可以用两个 3x3 卷积核堆叠来达到同样效果：

```
输入 → 3x3 Conv → 3x3 Conv → 输出
```

两个 3x3 的感受野 = 一个 5x5，但有三个优势：

1. **参数更少**：两个 3x3 = 2×3×3 = 18 参数，一个 5x5 = 25 参数
2. **非线性更强**：多了一次 ReLU，表达能力更好
3. **更统一**：全部用 3x3，网络结构更规整

### VGG 的结构

全部用 3x3 卷积核 + 2x2 池化，通过堆叠层数来增加深度：

| 网络 | 层数 | 卷积层数 |
|------|------|----------|
| VGG-11 | 11 | 8 |
| VGG-16 | 16 | 13 |
| VGG-19 | 19 | 16 |

In [2]:
# 两个 3x3 卷积 = 一个 5x5 感受野

x = torch.randn(1, 1, 16, 16)

# 方式1：一个 5x5 卷积
conv5x5 = nn.Conv2d(1, 1, 5, padding=2)
out1 = conv5x5(x)
params_5x5 = sum(p.numel() for p in conv5x5.parameters())

# 方式2：两个 3x3 卷积
conv3x3_a = nn.Conv2d(1, 1, 3, padding=1)
conv3x3_b = nn.Conv2d(1, 1, 3, padding=1)
out2 = conv3x3_b(conv3x3_a(x))
params_3x3 = sum(p.numel() for p in conv3x3_a.parameters()) + sum(p.numel() for p in conv3x3_b.parameters())

print(f"输入: {x.shape}")
print(f"5x5 卷积输出: {out1.shape}, 参数: {params_5x5}")
print(f"两个 3x3 输出: {out2.shape}, 参数: {params_3x3}")
print(f"\n参数节省: {params_5x5 - params_3x3} 个 ({(1 - params_3x3/params_5x5)*100:.0f}%)")

输入: torch.Size([1, 1, 16, 16])
5x5 卷积输出: torch.Size([1, 1, 16, 16]), 参数: 26
两个 3x3 输出: torch.Size([1, 1, 16, 16]), 参数: 20

参数节省: 6 个 (23%)


## 2. 用 PyTorch 搭建 VGG-16

VGG-16 的结构（针对 CIFAR-10 调整了最后的全连接层）：

```
64通道 3x3×2 → MaxPool → 128通道 3x3×2 → MaxPool → 256通道 3x3×3 → MaxPool → 512通道 3x3×3 → MaxPool → 512通道 3x3×3 → MaxPool → FC
```

In [3]:
class VGG16(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        # VGG 的卷积部分：全部用 3x3 卷积 + ReLU
        self.features = nn.Sequential(
            # Block 1: 3→64 通道
            nn.Conv2d(3, 64, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),              # 32x32 → 16x16

            # Block 2: 64→128 通道
            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(128, 128, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),              # 16x16 → 8x8

            # Block 3: 128→256 通道
            nn.Conv2d(128, 256, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(256, 256, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(256, 256, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),              # 8x8 → 4x4

            # Block 4: 256→512 通道
            nn.Conv2d(256, 512, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(512, 512, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(512, 512, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),              # 4x4 → 2x2

            # Block 5: 512→512 通道
            nn.Conv2d(512, 512, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(512, 512, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(512, 512, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),              # 2x2 → 1x1
        )
        # 全连接分类器
        self.classifier = nn.Sequential(
            nn.Linear(512 * 1 * 1, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x

model_vgg = VGG16().to(device)
print(f"VGG-16 参数量: {sum(p.numel() for p in model_vgg.parameters()):,}")

VGG-16 参数量: 14,982,474


## 3. Batch Normalization（批归一化）

### 问题：为什么深层网络训练难？

网络越深，每一层的输入分布会随着前面层的参数更新而不断变化（内部协变量偏移），导致：
- 需要更小的学习率
- 训练很慢，容易梯度消失/爆炸

### BN 的做法

对每个 mini-batch 内的数据做标准化，然后用可学习的参数 γ 和 β 做缩放和偏移：

```
1. 算 batch 内的均值和方差
2. 标准化：x_hat = (x - mean) / sqrt(var + eps)
3. 缩放偏移：y = γ * x_hat + β
```

γ 和 β 是可学习参数，让网络自己决定最优的分布。

### BN 的好处

- 训练更快，可以用更大学习率
- 有轻微正则化效果（因为每个 batch 的统计量有噪声）
- 减少对参数初始化的敏感度

### BN 放在哪里？

通常放在卷积之后、激活函数之前：`Conv → BN → ReLU`

In [4]:
# BN 演示
x = torch.randn(64, 32, 16, 16)  # batch=64, 32通道, 16x16

bn = nn.BatchNorm2d(32)  # 对 32 个通道分别做 BN
y = bn(x)

print(f"输入: {x.shape}")
print(f"输出: {y.shape}")
print(f"\nBN 可学习参数:")
print(f"  gamma (缩放): {bn.weight.shape}")
print(f"  beta  (偏移): {bn.bias.shape}")
print(f"  都是 32 维，每个通道一组")

# 验证：每个通道的输出近似均值为 0、方差为 1
print(f"\n输出各通道均值（近似0）: {y.mean(dim=[0,2,3])[:5].tolist()}")
print(f"输出各通道方差（近似1）: {y.var(dim=[0,2,3])[:5].tolist()}")

输入: torch.Size([64, 32, 16, 16])
输出: torch.Size([64, 32, 16, 16])

BN 可学习参数:
  gamma (缩放): torch.Size([32])
  beta  (偏移): torch.Size([32])
  都是 32 维，每个通道一组

输出各通道均值（近似0）: [-6.984919309616089e-10, -4.656612873077393e-10, 1.1641532182693481e-10, 8.731149137020111e-11, 1.979060471057892e-09]
输出各通道方差（近似1）: [1.0000510215759277, 1.0000511407852173, 1.0000509023666382, 1.0000507831573486, 1.0000509023666382]


## 4. VGG + BN：更现代的写法

在 VGG 的每个卷积后面加上 BN，就是更实用的版本。

In [5]:
class VGG16BN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(3, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # Block 2
            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # Block 3
            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.Conv2d(256, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.Conv2d(256, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # Block 4
            nn.Conv2d(256, 512, 3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.Conv2d(512, 512, 3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.Conv2d(512, 512, 3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # Block 5
            nn.Conv2d(512, 512, 3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.Conv2d(512, 512, 3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.Conv2d(512, 512, 3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x

model_vgg_bn = VGG16BN().to(device)
print(f"VGG-16 + BN 参数量: {sum(p.numel() for p in model_vgg_bn.parameters()):,}")

VGG-16 + BN 参数量: 14,990,922


## 5. ResNet：残差连接

### 问题：网络越深越好吗？

理论上更深的网络能学更复杂的特征。但实验发现：
- 56 层网络的训练误差反而比 20 层的高
- 这不是过拟合（训练集上也更差），而是**退化问题**
- 深层网络更难优化：梯度在反向传播中逐层衰减

### ResNet 的解决方案：残差连接（Skip Connection）

```
传统：输出 = F(x)          # 直接学习映射
ResNet：输出 = F(x) + x     # 学习残差
```

网络只需要学习「残差」F(x) = H(x) - x，即输入和输出之间的差异。

### 为什么这样更容易？

1. **恒等映射很容易学**：如果某一层不需要做任何变换，F(x) = 0 就行，输出直接等于输入
2. **梯度有捷径**：反向传播时梯度可以直接通过 skip connection 流回去，不会衰减
3. **类比**：就像考试时不用从零写答案，只需要在已有答案上做修改

### 残差块结构

```
x → Conv → BN → ReLU → Conv → BN → (+x) → ReLU
↓                                          ↑
└──────── skip connection ─────────────────┘
```

In [6]:
class ResidualBlock(nn.Module):
    """基本残差块：两个 3x3 卷积 + skip connection"""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)

        # 如果输入输出通道数不同，需要 1x1 卷积调整
        self.shortcut = nn.Sequential()
        if in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1),  # 1x1 卷积调整通道数
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        residual = self.shortcut(x)       # skip connection
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = self.relu(out + residual)   # 残差相加
        return out

# 测试
block = ResidualBlock(64, 128).to(device)
x = torch.randn(1, 64, 32, 32).to(device)
out = block(x)
print(f"输入: {x.shape}")
print(f"输出: {out.shape}")
print(f"参数: {sum(p.numel() for p in block.parameters()):,}")

输入: torch.Size([1, 64, 32, 32])
输出: torch.Size([1, 128, 32, 32])
参数: 230,528


## 6. 搭建 ResNet-18

ResNet-18 的结构：

```
7x7 Conv → MaxPool → 4组残差块（每组2个）→ Global AvgPool → FC
```

通道数变化：64 → 128 → 256 → 512

In [8]:
class ResNet18(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        # 初始卷积层
        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 64, 7, stride=2, padding=3),  # 32x32 → 16x16
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),                            # 16x16 → 8x8
        )

        # 4 组残差块
        self.layer1 = self._make_layer(64, 64, 2)    # 8x8
        self.layer2 = self._make_layer(64, 128, 2)   # 8x8
        self.layer3 = self._make_layer(128, 256, 2)  # 8x8
        self.layer4 = self._make_layer(256, 512, 2)  # 8x8

        # 全局平均池化 + 分类器
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))  # 任意大小 → 1x1
        self.fc = nn.Linear(512, num_classes)

    def _make_layer(self, in_channels, out_channels, num_blocks):
        """创建一组残差块"""
        layers = [ResidualBlock(in_channels, out_channels)]
        for _ in range(num_blocks - 1):
            layers.append(ResidualBlock(out_channels, out_channels))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.conv1(x)          # 初始卷积
        x = self.layer1(x)         # 残差块组
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)        # 全局平均池化
        x = x.view(x.size(0), -1)  # 展平
        x = self.fc(x)             # 分类
        return x

model_resnet = ResNet18().to(device)
print(f"ResNet-18 参数量: {sum(p.numel() for p in model_resnet.parameters()):,}")

ResNet-18 参数量: 11,186,442


## 7. 对比三种架构

用同一个数据形状，看三种网络的参数量和数据流变化。

In [9]:
x = torch.randn(1, 3, 32, 32).to(device)

models = {
    "简单CNN": model_vgg,     # 之前 03-cifar10 里的模型
    "VGG-16+BN": model_vgg_bn,
    "ResNet-18": model_resnet,
}

print(f"{'模型':<15} {'参数量':>12} {'输出形状':>15}")
print("-" * 45)
for name, m in models.items():
    params = sum(p.numel() for p in m.parameters())
    out = m(x)
    print(f"{name:<15} {params:>10,} {str(list(out.shape)):>15}")

模型                       参数量            输出形状
---------------------------------------------
简单CNN           14,982,474         [1, 10]
VGG-16+BN       14,990,922         [1, 10]
ResNet-18       11,186,442         [1, 10]


### 关键区别

| | 简单 CNN | VGG-16 | ResNet-18 |
|---|---------|--------|----------|
| 卷积核 | 3x3 | 全部 3x3 | 7x7 + 3x3 |
| 深度 | 3 层 | 13 层 | 18 层 |
| BN | 无 | 有 | 有 |
| Skip Connection | 无 | 无 | 有 |
| 池化 | MaxPool | MaxPool | MaxPool + Global AvgPool |
| 参数量 | ~300K | ~15M | ~11M |
| CIFAR-10 预期 | 70-80% | 85-90% | 90%+ |

## 8. 补充：Global Average Pooling

ResNet 用 `nn.AdaptiveAvgPool2d((1, 1))` 替代了传统的 Flatten + FC，这是现代 CNN 的常见做法：

- Flatten：把整个特征图拉成一维向量，参数量巨大
- Global AvgPool：对每个通道取平均值，直接得到一个数

好处：参数量大幅减少，减少过拟合。

In [10]:
# 对比 Flatten vs Global AvgPool
x = torch.randn(1, 512, 8, 8)

flatten_size = 512 * 8 * 8
gap_size = 512

print(f"特征图: {x.shape}")
print(f"Flatten 后: {flatten_size} 维")
print(f"Global AvgPool 后: {gap_size} 维")
print(f"\n如果接一个 FC 层到 10 类:")
print(f"  Flatten: {flatten_size * 10 + 10:,} 参数")
print(f"  GAP:     {gap_size * 10 + 10:,} 参数")

# PyTorch 演示
gap = nn.AdaptiveAvgPool2d((1, 1))
out = gap(x)
print(f"\nGAP 输入: {x.shape}")
print(f"GAP 输出: {out.shape}")  # (1, 512, 1, 1)

特征图: torch.Size([1, 512, 8, 8])
Flatten 后: 32768 维
Global AvgPool 后: 512 维

如果接一个 FC 层到 10 类:
  Flatten: 327,690 参数
  GAP:     5,130 参数

GAP 输入: torch.Size([1, 512, 8, 8])
GAP 输出: torch.Size([1, 512, 1, 1])


## 总结

### VGG 贡献
- 全部用 3x3 小卷积核堆叠，比大卷积核更高效
- 结构统一规整，容易理解和实现

### ResNet 贡献
- 残差连接解决了深层网络的退化问题
- 让网络可以训练到 100+ 层甚至 1000+ 层
- Skip connection 的思想影响了后续所有架构（包括 Transformer）

### BatchNorm 贡献
- 加速训练，允许更大学习率
- 现在几乎是所有网络的标配

这些思想组合在一起，就是现代 CNN 的基础。下一篇用预训练 ResNet-50 做迁移学习。